# Stream 2 Feature Extraction Pipeline

**Input:** RetinaFace crop images (already preprocessed, in `clip_dir` folders)

**This notebook does everything in order:**
1. Extract 478 MediaPipe landmarks from each frame → save `_landmarks_xyz.csv` per clip
2. Compute 38 Euclidean distance pairs (YOUR handpicked pairs) → save `_distances.csv` per clip
3. Compute 20 asymmetry features (abs diff + ratio) → save `_asymmetry.csv` per clip
4. Combine into task-wise CSVs (BIGSMILE, BLOW, etc.)
5. Combine into one overall CSV for Stream 2
6. Save `stream2_features.npy` per clip for the deep learning model

**NO re-running of face detection.** We read the existing RetinaFace crops directly.

In [1]:
import sys
print(sys.executable)

import mediapipe as mp
print(mp.__version__)

c:\Users\Nicola Sambo\anaconda3\envs\face\python.exe
0.10.14


---
## 1. Setup & Configuration

In [2]:
import cv2
import numpy as np
import pandas as pd
import math
import os
from pathlib import Path
from tqdm.notebook import tqdm
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# ======================== CONFIG — UPDATE THESE ========================
MANIFEST = r"C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\fusion\manifest_rf_crops_flat_fixed.csv"
OUTPUT_DIR = r"C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\fusion\STREAM2_FEATURES"
LANDMARKER_MODEL = r"C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\face_landmarker.task"
# ======================================================================

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load manifest
df = pd.read_csv(MANIFEST)
print(f"Manifest loaded: {len(df)} clips")
print(f"  HC: {(df['label']==0).sum()}, Stroke: {(df['label']==1).sum()}")
print(f"  Exercises: {sorted(df['exercise'].unique())}")
print(f"  Subjects: {df['subject_id'].nunique()}")
print(f"  Output dir: {OUTPUT_DIR}")

Manifest loaded: 185 clips
  HC: 80, Stroke: 105
  Exercises: ['BBP', 'BIG_SMILE', 'BLOW', 'BROW', 'KISS', 'OPEN', 'PA', 'PATAKA', 'SPREAD']
  Subjects: 26
  Output dir: C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\fusion\STREAM2_FEATURES


---
## 2. Define YOUR Landmark Pairs & Asymmetry Pairs

In [3]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  YOUR 38 LANDMARK DISTANCE PAIRS — your original design            ║
# ║  Not copied from any prior paper — this is YOUR novel selection     ║
# ╚══════════════════════════════════════════════════════════════════════╝

landmark_pairs = [
    ("AI",   70,  61),   # left eyebrow / upper cheek region to left mouth corner
    ("DJ",  300, 291),   # right eyebrow / upper cheek region to right mouth corner

    ("BI",  107,  61),   # left inner eyebrow region to left mouth corner
    ("CJ",  336, 291),   # right inner eyebrow region to right mouth corner

    ("AK",   70,  17),   # left eyebrow / upper cheek region to lower lip / chin midline
    ("DK",  300,  17),   # right eyebrow / upper cheek region to lower lip / chin midline

    ("BK",  107,  17),   # left inner eyebrow region to lower lip / chin midline
    ("CK",  336,  17),   # right inner eyebrow region to lower lip / chin midline

    ("FK",  133,  17),   # left inner eye corner to lower lip / chin midline
    ("GK",  362,  17),   # right inner eye corner to lower lip / chin midline

    ("EK",  130,  17),   # left outer eye corner to lower lip / chin midline
    ("HK",  359,  17),   # right outer eye corner to lower lip / chin midline

    ("AM",   70,  10),   # left eyebrow / upper cheek region to upper face midline
    ("MD",   10, 300),   # upper face midline to right eyebrow / upper cheek region

    ("EM",  130,  10),   # left outer eye corner to upper face midline
    ("HM",  359,  10),   # right outer eye corner to upper face midline

    ("EF_L", 130, 133),  # left eye width: outer eye corner to inner eye corner
    ("GH_R", 362, 359),  # right eye width: inner eye corner to outer eye corner

    ("PO_L", 472, 170),  # left iris/eye landmark to left cheek/eye-adjacent region
    ("QR_R", 475, 170),  # right iris/eye landmark to central cheek/eye-adjacent region

    ("BM",  107,  10),   # left inner eyebrow region to upper face midline
    ("CM",  336,  10),   # right inner eyebrow region to upper face midline

    ("IN",   61,   4),   # left mouth corner to nose tip
    ("JN",  291,   4),   # right mouth corner to nose tip

    ("FM",  133,  10),   # left inner eye corner to upper face midline
    ("GM",  362,  10),   # right inner eye corner to upper face midline

    ("IM",   61,  10),   # left mouth corner to upper face midline
    ("JM",  291,  10),   # right mouth corner to upper face midline

    ("FI",  133,  61),   # left inner eye corner to left mouth corner
    ("JG",  362, 291),   # right inner eye corner to right mouth corner

    ("FL",  133,   0),   # left inner eye corner to upper lip / mouth midline
    ("GL",  362,   0),   # right inner eye corner to upper lip / mouth midline

    ("IL",   61,   0),   # left mouth corner to upper lip / mouth midline
    ("JL",  291,   0),   # right mouth corner to upper lip / mouth midline

    ("EI",  130,  61),   # left outer eye corner to left mouth corner
    ("HJ",  359, 291),   # right outer eye corner to right mouth corner

    ("BL",  107,   0),   # left inner eyebrow region to upper lip / mouth midline
    ("CL",  336,   0),   # right inner eyebrow region to upper lip / mouth midline
]

# YOUR 20 ASYMMETRY PAIRS — matching your existing CSV format (D0, D1 ... D19)
asymmetry_pairs = [
    ("AI",   "DJ",   "D0"),
    ("BI",   "CJ",   "D1"),
    ("AK",   "DK",   "D2"),
    ("BK",   "CK",   "D3"),
    ("FK",   "GK",   "D4"),
    ("EK",   "HK",   "D5"),
    ("AM",   "MD",   "D6"),
    ("EM",   "HM",   "D7"),
    ("EF_L", "GH_R", "D8"),
    ("PO_L", "QR_R", "D9"),
    ("BM",   "CM",   "D10"),
    ("IN",   "JN",   "D11"),
    ("FM",   "GM",   "D12"),
    ("IM",   "JM",   "D14"),
    ("FI",   "JG",   "D15"),
    ("FL",   "GL",   "D16"),
    ("IL",   "JL",   "D17"),
    ("EI",   "HJ",   "D18"),
    ("BL",   "CL",   "D19"),
]

# All unique landmark IDs needed (for XYZ extraction)
ALL_IDS_NEEDED = sorted(set(
    [idx for _, i, j in landmark_pairs for idx in (i, j)] + [133, 362]  # + normalization pair
))

# Stable column orders
distance_labels = [lbl for lbl, _, _ in landmark_pairs]  # preserves your declaration order
asym_diff_labels = [name for _, _, name in asymmetry_pairs]  # DO, D1, ..., D19
asym_ratio_labels = [name + "_ratio" for _, _, name in asymmetry_pairs]
TOTAL_STREAM2_FEATURES = len(landmark_pairs) + len(asymmetry_pairs) * 2  # 38+20+20=78
print(f"Distance pairs : {len(landmark_pairs)}")
print(f"Asymmetry pairs: {len(asymmetry_pairs)}")
print(f"Unique landmark IDs needed: {len(ALL_IDS_NEEDED)}")
print(f"\nStream 2 feature vector per frame:")
print(f"  38 distances + 20 abs_diffs + 20 ratios = 76 features")

Distance pairs : 38
Asymmetry pairs: 19
Unique landmark IDs needed: 17

Stream 2 feature vector per frame:
  38 distances + 20 abs_diffs + 20 ratios = 76 features


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CLINICAL VISUALIZATION — 38 Landmark Pairs on a Face Frame            ║
# ║  Self-contained: run AFTER ef571a6c + define-pairs cells only           ║
# ║  Shows which distances are measured and their normalized values          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import cv2, math, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import display, Image as IPImage
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# ── Initialize MediaPipe (self-contained, does not need helpers cell) ──
_base = mp_python.BaseOptions(model_asset_path=LANDMARKER_MODEL)
_opts = mp_vision.FaceLandmarkerOptions(
    base_options=_base,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)
_det = mp_vision.FaceLandmarker.create_from_options(_opts)
print("MediaPipe initialized.")

# ── Load one sample frame from the first clip in the manifest ──
_clip_dir = Path(df.iloc[0]["clip_dir"])
_frames   = sorted([f for f in _clip_dir.iterdir()
                    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
assert _frames, f"No images found in {_clip_dir}"

_img_bgr = cv2.imread(str(_frames[0]))
_h, _w   = _img_bgr.shape[:2]
_img_rgb = cv2.cvtColor(_img_bgr, cv2.COLOR_BGR2RGB)
print(f"Frame: {_frames[0].name}  |  size: {_w}x{_h}  |  "
      f"subject={df.iloc[0]['subject_id']}  exercise={df.iloc[0]['exercise']}")

# ── Run face landmark detection ──
_mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=_img_rgb)
_res    = _det.detect(_mp_img)
assert _res.face_landmarks, "No face detected — edit df.iloc[0] to pick a different clip."
_lm = _res.face_landmarks[0]

# ── Intercanthal distance (ICD) for normalization ──
def _eu3(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)

_L_eye = (_lm[133].x*_w, _lm[133].y*_h, _lm[133].z*_w)
_R_eye = (_lm[362].x*_w, _lm[362].y*_h, _lm[362].z*_w)
_ICD   = _eu3(_L_eye, _R_eye) or 1.0

# ═══════════════════════════════════════════════════════════════
#  PANEL A — Face image with all 38 pairs drawn
# ═══════════════════════════════════════════════════════════════
MIDLINE_IDS  = {10, 4, 0, 17}
C_LEFT    = (220,  60,   0)   # blue  (BGR)
C_RIGHT   = (  0,  60, 220)   # red
C_MIDLINE = (  0, 210, 210)   # cyan-yellow
C_DOT     = (  0, 210,   0)   # green

vis = _img_bgr.copy()
dist_table = []   # (label, idx1, idx2, dist_norm, color)

for pair_lbl, idx1, idx2 in landmark_pairs:
    lm1, lm2 = _lm[idx1], _lm[idx2]
    pt1 = (int(lm1.x*_w), int(lm1.y*_h))
    pt2 = (int(lm2.x*_w), int(lm2.y*_h))
    p1_3 = (lm1.x*_w, lm1.y*_h, lm1.z*_w)
    p2_3 = (lm2.x*_w, lm2.y*_h, lm2.z*_w)
    d = round(_eu3(p1_3, p2_3) / _ICD, 3)

    if idx1 in MIDLINE_IDS or idx2 in MIDLINE_IDS:
        col = C_MIDLINE
    elif idx1 < 300:
        col = C_LEFT
    else:
        col = C_RIGHT

    cv2.line(vis, pt1, pt2, col, 2, cv2.LINE_AA)

    # distance label at midpoint
    mx, my = (pt1[0]+pt2[0])//2, (pt1[1]+pt2[1])//2
    cv2.putText(vis, f"{pair_lbl}", (mx+2, my-2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.28, (255,255,255), 1, cv2.LINE_AA)

    dist_table.append((pair_lbl, idx1, idx2, d, col))

# Landmark dots
_sel_ids = sorted({idx for _, i, j in landmark_pairs for idx in (i, j)})
for idx in _sel_ids:
    x, y = int(_lm[idx].x*_w), int(_lm[idx].y*_h)
    cv2.circle(vis, (x, y), 5, C_DOT, -1)
    cv2.circle(vis, (x, y), 5, (0, 0, 0), 1)
    cv2.putText(vis, str(idx), (x+5, y-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.28, (220,220,220), 1, cv2.LINE_AA)

# Legend
_legend = [(C_LEFT,"Left-side pairs"),(C_RIGHT,"Right-side pairs"),
           (C_MIDLINE,"Midline connections"),(C_DOT,"Landmark nodes")]
for i, (lc, lt) in enumerate(_legend):
    yp = 20 + i*22
    cv2.circle(vis, (12, yp), 7, lc, -1)
    cv2.putText(vis, lt, (24, yp+5), cv2.FONT_HERSHEY_SIMPLEX, 0.42, lc, 1, cv2.LINE_AA)

# Footer bar with title
cv2.rectangle(vis, (0, _h-28), (_w, _h), (20, 20, 20), -1)
cv2.putText(vis,
    f"38 Clinical Landmark Pairs  |  ICD={_ICD:.1f}px  |  "
    f"{df.iloc[0]['subject_id']} · {df.iloc[0]['exercise']} · {df.iloc[0]['label_str']}",
    (5, _h-9), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200,200,200), 1, cv2.LINE_AA)

_out_dir  = r"C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\stream2_debug_from_video"
os.makedirs(_out_dir, exist_ok=True)
_face_png = os.path.join(_out_dir, "A_clinical_pairs_face.png")
cv2.imwrite(_face_png, vis)

# ═══════════════════════════════════════════════════════════════
#  PANEL B — Bar chart of all 38 normalized distances
# ═══════════════════════════════════════════════════════════════
labels_b = [r[0] for r in dist_table]
dists_b  = [r[3] for r in dist_table]
# Convert BGR color to matplotlib RGB fraction
def _bgr2rgb(c): return (c[2]/255, c[1]/255, c[0]/255)
colors_b = [_bgr2rgb(r[4]) for r in dist_table]

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(labels_b, dists_b, color=colors_b, edgecolor='white', linewidth=0.5)
ax.set_xlabel("Landmark Pair", fontsize=11)
ax.set_ylabel("Normalized Distance (ICD units)", fontsize=11)
ax.set_title(
    f"38 Clinical Landmark Distances — {df.iloc[0]['subject_id']} · "
    f"{df.iloc[0]['exercise']} · {df.iloc[0]['label_str']}  "
    f"(frame 0, normalized by intercanthal distance)",
    fontsize=11)
ax.tick_params(axis='x', rotation=75, labelsize=7)
ax.set_ylim(0, max(dists_b)*1.15)
for bar, val in zip(bars, dists_b):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f"{val:.2f}", ha='center', va='bottom', fontsize=6.5)
_patches = [
    mpatches.Patch(color=_bgr2rgb(C_LEFT),    label='Left-side pairs'),
    mpatches.Patch(color=_bgr2rgb(C_RIGHT),   label='Right-side pairs'),
    mpatches.Patch(color=_bgr2rgb(C_MIDLINE), label='Midline connections'),
]
ax.legend(handles=_patches, fontsize=9)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
_bar_png = os.path.join(_out_dir, "B_clinical_pairs_barchart.png")
fig.savefig(_bar_png, dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════
#  PANEL C — Asymmetry table (|Left − Right| per D0-D19)
# ═══════════════════════════════════════════════════════════════
dist_lookup = {r[0]: r[3] for r in dist_table}
print(f"\n{'─'*62}")
print(f"  Asymmetry features (frame 0, normalized by ICD={_ICD:.1f}px)")
print(f"{'─'*62}")
print(f"  {'ID':<6}  {'Left pair':<8}  {'Right pair':<8}  {'|L|':>6}  {'|R|':>6}  {'|L−R|':>7}  {'ratio':>7}")
print(f"  {'─'*60}")
for Ll, Rl, name in asymmetry_pairs:
    Lv = dist_lookup.get(Ll, 0)
    Rv = dist_lookup.get(Rl, 0)
    diff  = abs(Lv - Rv)
    avg   = (Lv + Rv) / 2.0
    ratio = diff / avg if avg > 0 else 0
    flag = " ◄ HIGH" if diff > 0.15 else ""
    print(f"  {name:<6}  {Ll:<8}  {Rl:<8}  {Lv:>6.3f}  {Rv:>6.3f}  {diff:>7.3f}  {ratio:>7.3f}{flag}")

print(f"\nSaved:")
print(f"  Face image : {_face_png}")
print(f"  Bar chart  : {_bar_png}")
display(IPImage(filename=_face_png, width=680))

In [4]:
#Step 3: Debug one clip/frame
from pathlib import Path
import os

BASE_DIR = Path(r"C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps")

video_exts = {".mp4", ".avi", ".mov", ".mkv"}
image_exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

video_files = []
image_files = []

for p in BASE_DIR.rglob("*"):
    if p.suffix in video_exts:
        video_files.append(p)
    elif p.suffix in image_exts:
        image_files.append(p)

print("Video files found:", len(video_files))
print("Image frames found:", len(image_files))

print("\nFirst 5 videos:")
for v in video_files[:5]:
    print(v)

print("\nFirst 5 images:")
for img in image_files[:5]:
    print(img)


Video files found: 1
Image frames found: 126934

First 5 videos:
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\OP01_02_BBP_NORMAL_color.avi

First 5 images:
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\bounding_box.png
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Detection coverage.png
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Interpretation and decision.png
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Cropped_Toronto_Files\Cropped_Toronto_Files\HC\BBP\N001_02_BBP_NORMAL_color\frame_0001\mediapipe_bbox.jpg
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FA

In [5]:
import cv2

DEBUG_DIR = BASE_DIR / "stream2_debug_from_video"
FRAME_DIR = DEBUG_DIR / "frames"
FRAME_DIR.mkdir(parents=True, exist_ok=True)

sample_video = video_files[0]
print("Using video:", sample_video)

cap = cv2.VideoCapture(str(sample_video))

frame_count = 0
saved_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # save every frame, or change to frame_count % 5 == 0 if too many
    out_path = FRAME_DIR / f"frame_{saved_count:04d}.jpg"
    cv2.imwrite(str(out_path), frame)

    saved_count += 1
    frame_count += 1

cap.release()

print("Total frames read:", frame_count)
print("Frames saved:", saved_count)
print("Saved to:", FRAME_DIR)

Using video: C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\OP01_02_BBP_NORMAL_color.avi
Total frames read: 2758
Frames saved: 2758
Saved to: C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\stream2_debug_from_video\frames


In [6]:
# Find folders that contain image frames
frame_folders = sorted(set([p.parent for p in image_files]))

print("Frame folders found:", len(frame_folders))
for f in frame_folders[:10]:
    print(f)

Frame folders found: 15709
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Cropped_Toronto_Files\Cropped_Toronto_Files\HC\BBP\N001_02_BBP_NORMAL_color\frame_0001
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Cropped_Toronto_Files\Cropped_Toronto_Files\HC\BBP\N001_02_BBP_NORMAL_color\frame_0002
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Cropped_Toronto_Files\Cropped_Toronto_Files\HC\BBP\N001_02_BBP_NORMAL_color\frame_0003
C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Cropped_Toronto_Files\Cropped_Toronto_Files\HC\BBP\N001_02_BBP_NORM

In [7]:
sample_clip_dir = str(frame_folders[0])
print("Using clip folder:", sample_clip_dir)

Using clip folder: C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps


In [ ]:
# ── DEBUG: Load a sample cropped frame and run MediaPipe detection ──
# Fix: image_rgb was never defined — load it here first

from pathlib import Path

# Pick the first frame from the first clip in the manifest
sample_clip_dir = Path(df.iloc[0]["clip_dir"])
sample_frames = sorted([f for f in sample_clip_dir.iterdir()
                         if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])
assert sample_frames, f"No images in {sample_clip_dir}"
sample_path = sample_frames[0]
print(f"Using frame: {sample_path}")

# Load image and convert to RGB
image_bgr = cv2.imread(str(sample_path))
h, w = image_bgr.shape[:2]
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
print(f"Image size: {w}x{h}")

# Run detection using face_mesh already initialized in the helpers cell
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
result = face_mesh.detect(mp_image)

assert result.face_landmarks, "No face detected — try a different frame"
landmarks = result.face_landmarks[0]
print(f"Detected {len(landmarks)} landmarks")


In [ ]:
# ── VISUALIZE: Draw landmark nodes + lines between all 38 clinical pairs ──
# Requires cells run in order: define-pairs → helpers → cell above

import os
from IPython.display import display, Image as IPImage

MIDLINE_IDS = {10, 4, 0, 17}   # landmarks on the face midline

vis = image_bgr.copy()

# Draw a colored line for each of the 38 landmark pairs
for pair_label, idx1, idx2 in landmark_pairs:
    lm1, lm2 = landmarks[idx1], landmarks[idx2]
    pt1 = (int(lm1.x * w), int(lm1.y * h))
    pt2 = (int(lm2.x * w), int(lm2.y * h))

    if idx2 in MIDLINE_IDS or idx1 in MIDLINE_IDS:
        color = (0, 220, 220)   # yellow  — midline connections
    elif idx1 < 300:            # left-side source
        color = (255, 80, 0)    # blue    — left-side pairs
    else:                       # right-side source
        color = (0, 80, 255)    # red     — right-side pairs

    cv2.line(vis, pt1, pt2, color, 1, cv2.LINE_AA)

# Draw filled dot + ID label for every selected node
selected_ids = sorted(set([idx for _, i, j in landmark_pairs for idx in (i, j)]))
for idx in selected_ids:
    lm = landmarks[idx]
    x, y = int(lm.x * w), int(lm.y * h)
    cv2.circle(vis, (x, y), 4, (0, 255, 0), -1)
    cv2.putText(vis, str(idx), (x + 4, y - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1, cv2.LINE_AA)

# Legend
legend = [
    ((255, 80,  0), "Left-side pairs"),
    ((0,  80, 255), "Right-side pairs"),
    ((0, 220, 220), "Midline connections"),
    ((0, 255,   0), "Landmark nodes"),
]
for i, (col, label_text) in enumerate(legend):
    y_pos = 20 + i * 20
    cv2.circle(vis, (12, y_pos), 6, col, -1)
    cv2.putText(vis, label_text, (22, y_pos + 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1, cv2.LINE_AA)

# Save and display
out_dir = r"C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\stream2_debug_from_video"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "landmark_pairs_visualization.png")
cv2.imwrite(out_path, vis)
print(f"Saved: {out_path}")
display(IPImage(filename=out_path, width=600))


---
## 3. Helper Functions

In [ ]:
def list_images_sorted(folder):
    """List image files in folder, sorted by name."""
    try:
        names = sorted(f for f in os.listdir(folder) if Path(f).suffix in IMAGE_EXTS)
    except FileNotFoundError:
        names = []
    return [os.path.join(folder, f) for f in names]


def euclidean_3d(p1, p2):
    """3D Euclidean distance."""
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2 + (p1[2]-p2[2])**2)


# ── Initialize MediaPipe FaceLandmarker (Tasks API) ──
base_options = mp_python.BaseOptions(model_asset_path=LANDMARKER_MODEL)
options = mp_vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)
face_mesh = mp_vision.FaceLandmarker.create_from_options(options)

print("✅ MediaPipe FaceLandmarker initialized (Tasks API)")

✅ MediaPipe FaceLandmarker initialized (Tasks API)


In [ ]:
import sys
print(sys.executable)

c:\Users\Nicola Sambo\anaconda3\envs\face\python.exe


---
## 4. Process ONE Clip (test before batch)

For a single clip, this does all 3 steps:
1. Extract 478 landmarks → `_landmarks_xyz.csv`
2. Compute 38 distances → `_distances.csv`
3. Compute 20 asymmetries → `_abs_diff.csv` + `_abs_ratio.csv`

In [ ]:
def process_one_clip(clip_dir, subject_id, label, label_str, exercise, face_mesh):
    """
    Process all frames in one clip directory.
    
    Input:  RetinaFace crop images in clip_dir
    
    Returns dict with:
        'xyz_df'       : DataFrame of selected landmark XYZ per frame
        'distances_df' : DataFrame of 38 normalized distances per frame
        'abs_diff_df'  : DataFrame of 20 asymmetry |L-R| per frame
        'abs_ratio_df' : DataFrame of 20 asymmetry ratios per frame
        'stream2_arr'  : numpy array (N_frames, 78) for the deep learning model
        'landmarks_arr': numpy array (N_frames, 478, 3) raw landmarks
    """
    frames = list_images_sorted(clip_dir)
    if not frames:
        return None
    
    # Storage
    xyz_rows = []
    distance_rows = []
    asym_rows = []
    all_landmarks_raw = []
    
    for frame_idx, fpath in enumerate(frames):
        image = cv2.imread(fpath)
        if image is None:
            # Append empty rows to keep frame indexing consistent
            xyz_rows.append({"frame": frame_idx})
            distance_rows.append({"frame": frame_idx})
            asym_rows.append({"frame": frame_idx})
            all_landmarks_raw.append(np.zeros((478, 3), dtype=np.float32))
            continue
        
        h, w = image.shape[:2]
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)
        
        xyz_row = {"frame": frame_idx}
        dist_row = {"frame": frame_idx}
        
        if results.multi_face_landmarks:
            lm = results.multi_face_landmarks[0].landmark
            
            # ── Save raw 478 landmarks ──
            raw_478 = np.array([[p.x, p.y, p.z] for p in lm], dtype=np.float32)
            all_landmarks_raw.append(raw_478)
            
            # ── Step 1: Extract XYZ for selected landmark IDs ──
            for idx in ALL_IDS_NEEDED:
                if idx < len(lm):
                    xyz_row[f"x_{idx}"] = lm[idx].x * w
                    xyz_row[f"y_{idx}"] = lm[idx].y * h
                    xyz_row[f"z_{idx}"] = lm[idx].z * w
                else:
                    xyz_row[f"x_{idx}"] = None
                    xyz_row[f"y_{idx}"] = None
                    xyz_row[f"z_{idx}"] = None
            
            # ── Step 2: Compute 38 normalized 3D distances ──
            # Intercanthal normalization (133 ↔ 362)
            left_eye  = (lm[133].x * w, lm[133].y * h, lm[133].z * w)
            right_eye = (lm[362].x * w, lm[362].y * h, lm[362].z * w)
            intercanthal = euclidean_3d(left_eye, right_eye) or 1.0
            
            for label_name, idx1, idx2 in landmark_pairs:
                if idx1 >= len(lm) or idx2 >= len(lm):
                    dist_row[label_name] = None
                    continue
                p1 = (lm[idx1].x * w, lm[idx1].y * h, lm[idx1].z * w)
                p2 = (lm[idx2].x * w, lm[idx2].y * h, lm[idx2].z * w)
                d = euclidean_3d(p1, p2)
                dist_row[label_name] = round(d / intercanthal, 5)
        else:
            # No face detected
            all_landmarks_raw.append(np.zeros((478, 3), dtype=np.float32))
            for idx in ALL_IDS_NEEDED:
                xyz_row[f"x_{idx}"] = None
                xyz_row[f"y_{idx}"] = None
                xyz_row[f"z_{idx}"] = None
            for label_name, _, _ in landmark_pairs:
                dist_row[label_name] = None
        
        xyz_rows.append(xyz_row)
        distance_rows.append(dist_row)
    
    # ── Step 3: Compute 20 asymmetry features from distances ──
    for dist_row in distance_rows:
        asym_row = {"frame": dist_row["frame"]}
        for left_lbl, right_lbl, asym_name in asymmetry_pairs:
            Lv = dist_row.get(left_lbl)
            Rv = dist_row.get(right_lbl)
            if Lv is not None and Rv is not None:
                diff = abs(Lv - Rv)
                avg = (Lv + Rv) / 2.0
                asym_row[asym_name] = round(diff, 5)
                asym_row[asym_name + "_ratio"] = round(diff / avg, 5) if avg > 0 else 0.0
            else:
                asym_row[asym_name] = None
                asym_row[asym_name + "_ratio"] = None
        asym_rows.append(asym_row)
    
    # ── Build DataFrames with stable column order ──
    xyz_cols = ["frame"] + [f"x_{i}" for i in ALL_IDS_NEEDED] + \
               [f"y_{i}" for i in ALL_IDS_NEEDED] + [f"z_{i}" for i in ALL_IDS_NEEDED]
    xyz_df = pd.DataFrame(xyz_rows).reindex(columns=xyz_cols)
    
    dist_cols = ["frame"] + distance_labels
    distances_df = pd.DataFrame(distance_rows).reindex(columns=dist_cols)
    
    abs_diff_cols = ["frame"] + asym_diff_labels
    abs_ratio_cols = ["frame"] + asym_ratio_labels
    asym_df = pd.DataFrame(asym_rows)
    abs_diff_df = asym_df.reindex(columns=abs_diff_cols)
    abs_ratio_df = asym_df.reindex(columns=abs_ratio_cols)
    
    # ── Build Stream 2 numpy array: 38 distances + 20 diffs + 20 ratios = 78 ──
    stream2_cols = distance_labels + asym_diff_labels + asym_ratio_labels
    combined = pd.merge(distances_df, asym_df, on="frame", how="left")
    stream2_arr = combined[stream2_cols].fillna(0).values.astype(np.float32)
    
    # Raw landmarks array
    landmarks_arr = np.stack(all_landmarks_raw)  # (N_frames, 478, 3)
    
    return {
        'xyz_df': xyz_df,
        'distances_df': distances_df,
        'abs_diff_df': abs_diff_df,
        'abs_ratio_df': abs_ratio_df,
        'stream2_arr': stream2_arr,      # (N_frames, 78)
        'landmarks_arr': landmarks_arr,  # (N_frames, 478, 3)
    }

print("✅ process_one_clip() defined.")

✅ process_one_clip() defined.


In [ ]:
def process_one_clip(clip_dir, subject_id, label, label_str, exercise, face_mesh):
    """
    Process all frames in one clip directory using MediaPipe Tasks API.
    
    Input:  RetinaFace crop images in clip_dir
    
    Returns dict with:
        'xyz_df'       : DataFrame of selected landmark XYZ per frame
        'distances_df' : DataFrame of 38 normalized distances per frame
        'abs_diff_df'  : DataFrame of 20 asymmetry |L-R| per frame
        'abs_ratio_df' : DataFrame of 20 asymmetry ratios per frame
        'stream2_arr'  : numpy array (N_frames, 78) for the deep learning model
        'landmarks_arr': numpy array (N_frames, 478, 3) raw landmarks
    """
    frames = list_images_sorted(clip_dir)
    if not frames:
        return None
    
    # Storage
    xyz_rows = []
    distance_rows = []
    asym_rows = []
    all_landmarks_raw = []
    
    for frame_idx, fpath in enumerate(frames):
        image = cv2.imread(fpath)
        if image is None:
            # Append empty rows to keep frame indexing consistent
            xyz_rows.append({"frame": frame_idx})
            distance_rows.append({"frame": frame_idx})
            asym_rows.append({"frame": frame_idx})
            all_landmarks_raw.append(np.zeros((478, 3), dtype=np.float32))
            continue
        
        h, w = image.shape[:2]
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Convert to MediaPipe Image and detect (Tasks API)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        results = face_mesh.detect(mp_image)
        
        xyz_row = {"frame": frame_idx}
        dist_row = {"frame": frame_idx}
        
        if results.face_landmarks:
            lm = results.face_landmarks[0]  # First face (normalized 0-1 coordinates)
            
            # ── Save raw 478 landmarks ──
            raw_478 = np.array([[p.x, p.y, p.z] for p in lm], dtype=np.float32)
            all_landmarks_raw.append(raw_478)
            
            # ── Step 1: Extract XYZ for selected landmark IDs ──
            for idx in ALL_IDS_NEEDED:
                if idx < len(lm):
                    xyz_row[f"x_{idx}"] = lm[idx].x * w
                    xyz_row[f"y_{idx}"] = lm[idx].y * h
                    xyz_row[f"z_{idx}"] = lm[idx].z * w
                else:
                    xyz_row[f"x_{idx}"] = None
                    xyz_row[f"y_{idx}"] = None
                    xyz_row[f"z_{idx}"] = None
            
            # ── Step 2: Compute 38 normalized 3D distances ──
            # Intercanthal normalization (133 ↔ 362)
            left_eye  = (lm[133].x * w, lm[133].y * h, lm[133].z * w)
            right_eye = (lm[362].x * w, lm[362].y * h, lm[362].z * w)
            intercanthal = euclidean_3d(left_eye, right_eye) or 1.0
            
            for label_name, idx1, idx2 in landmark_pairs:
                if idx1 >= len(lm) or idx2 >= len(lm):
                    dist_row[label_name] = None
                    continue
                p1 = (lm[idx1].x * w, lm[idx1].y * h, lm[idx1].z * w)
                p2 = (lm[idx2].x * w, lm[idx2].y * h, lm[idx2].z * w)
                d = euclidean_3d(p1, p2)
                dist_row[label_name] = round(d / intercanthal, 5)
        else:
            # No face detected
            all_landmarks_raw.append(np.zeros((478, 3), dtype=np.float32))
            for idx in ALL_IDS_NEEDED:
                xyz_row[f"x_{idx}"] = None
                xyz_row[f"y_{idx}"] = None
                xyz_row[f"z_{idx}"] = None
            for label_name, _, _ in landmark_pairs:
                dist_row[label_name] = None
        
        xyz_rows.append(xyz_row)
        distance_rows.append(dist_row)
    
    # ── Step 3: Compute 20 asymmetry features from distances ──
    for dist_row in distance_rows:
        asym_row = {"frame": dist_row["frame"]}
        for left_lbl, right_lbl, asym_name in asymmetry_pairs:
            Lv = dist_row.get(left_lbl)
            Rv = dist_row.get(right_lbl)
            if Lv is not None and Rv is not None:
                diff = abs(Lv - Rv)
                avg = (Lv + Rv) / 2.0
                asym_row[asym_name] = round(diff, 5)
                asym_row[asym_name + "_ratio"] = round(diff / avg, 5) if avg > 0 else 0.0
            else:
                asym_row[asym_name] = None
                asym_row[asym_name + "_ratio"] = None
        asym_rows.append(asym_row)
    
    # ── Build DataFrames with stable column order ──
    xyz_cols = ["frame"] + [f"x_{i}" for i in ALL_IDS_NEEDED] + \
               [f"y_{i}" for i in ALL_IDS_NEEDED] + [f"z_{i}" for i in ALL_IDS_NEEDED]
    xyz_df = pd.DataFrame(xyz_rows).reindex(columns=xyz_cols)
    
    dist_cols = ["frame"] + distance_labels
    distances_df = pd.DataFrame(distance_rows).reindex(columns=dist_cols)
    
    abs_diff_cols = ["frame"] + asym_diff_labels
    abs_ratio_cols = ["frame"] + asym_ratio_labels
    asym_df = pd.DataFrame(asym_rows)
    abs_diff_df = asym_df.reindex(columns=abs_diff_cols)
    abs_ratio_df = asym_df.reindex(columns=abs_ratio_cols)
    
    # ── Build Stream 2 numpy array: 38 distances + 20 diffs + 20 ratios = 78 ──
    stream2_cols = distance_labels + asym_diff_labels + asym_ratio_labels
    combined = pd.merge(distances_df, asym_df, on="frame", how="left")
    stream2_arr = combined[stream2_cols].fillna(0).values.astype(np.float32)
    
    # Raw landmarks array
    landmarks_arr = np.stack(all_landmarks_raw)  # (N_frames, 478, 3)
    
    return {
        'xyz_df': xyz_df,
        'distances_df': distances_df,
        'abs_diff_df': abs_diff_df,
        'abs_ratio_df': abs_ratio_df,
        'stream2_arr': stream2_arr,
        'landmarks_arr': landmarks_arr,
    }

In [ ]:
# ── TEST on a single clip ──
test_row = df.iloc[0]
print(f"Testing on: {test_row['clip_dir']}")
print(f"  Exercise: {test_row['exercise']}, Label: {test_row['label_str']}, Subject: {test_row['subject_id']}")

result = process_one_clip(
    clip_dir=test_row['clip_dir'],
    subject_id=test_row['subject_id'],
    label=test_row['label'],
    label_str=test_row['label_str'],
    exercise=test_row['exercise'],
    face_mesh=face_mesh
)

print(f"\n  Results:")
print(f"    Landmarks XYZ   : {result['xyz_df'].shape}")
print(f"    Distances (38)  : {result['distances_df'].shape}")
print(f"    Asymmetry diffs : {result['abs_diff_df'].shape}")
print(f"    Asymmetry ratios: {result['abs_ratio_df'].shape}")
print(f"    Stream2 array   : {result['stream2_arr'].shape}  ← (frames, 78)")
print(f"    Raw landmarks   : {result['landmarks_arr'].shape}  ← (frames, 478, 3)")
print(f"\n  Sample distances (frame 0):")
print(result['distances_df'].iloc[0].to_string())
print(f"\n  Sample asymmetry (frame 0):")
print(result['abs_diff_df'].iloc[0].to_string())

Testing on: C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Cropped_Toronto_Files\Cropped_Toronto_Files\HC\BBP\N001_02_BBP_NORMAL_color\rf_crops_flat
  Exercise: BBP, Label: HC, Subject: N001

  Results:
    Landmarks XYZ   : (101, 52)
    Distances (38)  : (101, 39)
    Asymmetry diffs : (101, 20)
    Asymmetry ratios: (101, 20)
    Stream2 array   : (101, 76)  ← (frames, 78)
    Raw landmarks   : (101, 478, 3)  ← (frames, 478, 3)

  Sample distances (frame 0):
frame    0.00000
AI       2.63458
DJ       2.74894
BI       2.77721
CJ       2.90806
AK       3.31178
DK       3.37890
BK       3.07721
CK       3.12680
FK       2.45157
GK       2.41834
EK       2.75692
HK       2.74860
AM       1.80047
MD       2.06714
EM       1.95410
HM       2.12253
EF_L     0.79909
GH_R     0.88591
PO_L     2.73251
QR_R     3.52480
BM       0.88613
CM       0.91770
IN       1.69176
JN       1.72971
FM       1.55624
GM       1

---
## 5. Process ALL 185 Clips

Saves per-clip CSVs + task-wise combined + overall combined.

**Estimated time:** ~4-5 minutes for 15,522 frames.

In [ ]:
# ── Process ALL clips and save everything ──

# Storage for task-wise and overall combining
task_dfs = {"distances": {}, "abs_diff": {}, "abs_ratio": {}, "combined": {}}
stats = {"done": 0, "failed": 0}

# Create a dedicated output folder for processed features
FEATURES_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "per_clip_features")
os.makedirs(FEATURES_OUTPUT_DIR, exist_ok=True)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing clips"):
    clip_dir = row['clip_dir']
    subject_id = row['subject_id']
    label_int = row['label']
    label_str = row['label_str']
    exercise = row['exercise']
    clip_name = row['clip_name']
    
    # Create a unique output folder for this clip's features
    clip_features_dir = os.path.join(FEATURES_OUTPUT_DIR, f"{label_str}_{exercise}_{clip_name}")
    os.makedirs(clip_features_dir, exist_ok=True)
    
    result = process_one_clip(clip_dir, subject_id, label_int, label_str, exercise, face_mesh)
    
    if result is None:
        print(f"  ⚠️ Failed: {clip_dir}")
        stats['failed'] += 1
        continue
    
    # ── Save per-clip CSVs into clip_features_dir (NOT the original clip_dir) ──
    result['xyz_df'].to_csv(os.path.join(clip_features_dir, f"{clip_name}_landmarks_xyz.csv"), index=False)
    result['distances_df'].to_csv(os.path.join(clip_features_dir, f"{clip_name}_distances.csv"), index=False)
    result['abs_diff_df'].to_csv(os.path.join(clip_features_dir, f"{clip_name}_abs_diff.csv"), index=False)
    result['abs_ratio_df'].to_csv(os.path.join(clip_features_dir, f"{clip_name}_abs_ratio.csv"), index=False)
    
    # ── Save .npy files for the deep learning model ──
    np.save(os.path.join(clip_features_dir, 'stream2_features.npy'), result['stream2_arr'])
    np.save(os.path.join(clip_features_dir, 'landmarks_478.npy'), result['landmarks_arr'])
    
    # ── Collect for task-wise combining ──
    # Add metadata columns matching your existing CSV format
    for key, df_data in [("distances", result['distances_df']),
                          ("abs_diff", result['abs_diff_df']),
                          ("abs_ratio", result['abs_ratio_df'])]:
        df_copy = df_data.copy()
        df_copy['subject_id'] = subject_id
        df_copy['label'] = label_int
        df_copy['task'] = exercise
        
        if exercise not in task_dfs[key]:
            task_dfs[key][exercise] = []
        task_dfs[key][exercise].append(df_copy)
    
    # Combined (distances + asymmetry) for overall file
    # Create a proper combined dataframe
    combined_df = result['distances_df'].copy()
    for col in asym_diff_labels:
        if col in result['abs_diff_df'].columns:
            combined_df[col] = result['abs_diff_df'][col]
    for col in asym_ratio_labels:
        if col in result['abs_ratio_df'].columns:
            combined_df[col] = result['abs_ratio_df'][col]
    
    combined_df['subject_id'] = subject_id
    combined_df['label'] = label_int
    combined_df['task'] = exercise
    
    if exercise not in task_dfs['combined']:
        task_dfs['combined'][exercise] = []
    task_dfs['combined'][exercise].append(combined_df)
    
    stats['done'] += 1

print(f"\n✅ Processing complete!")
print(f"   Done: {stats['done']}, Failed: {stats['failed']}")

Processing clips:   0%|          | 0/185 [00:00<?, ?it/s]

  ⚠️ Failed: C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Cropped_Toronto_Files\Cropped_Toronto_Files\STROKE\SPREAD\OP02_02_NSM_SPREAD_color - Trim\rf_crops_flat

✅ Processing complete!
   Done: 184, Failed: 1


In [ ]:
# ── RETRY: Process the 1 failed clip (manifest path typo: "- Trim") ──
# The manifest entry says "OP02_02_NSM_SPREAD_color - Trim\rf_crops_flat"
# but the real folder is "OP02_02_NSM_SPREAD_color\rf_crops_flat"

CORRECT_CLIP_DIR = (
    r"C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING"
    r"\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\Cropped_Toronto_Files"
    r"\Cropped_Toronto_Files\STROKE\SPREAD\OP02_02_NSM_SPREAD_color\rf_crops_flat"
)

# Find the manifest row
failed_mask = df['clip_dir'].str.contains('Trim', na=False)
assert failed_mask.sum() == 1, f"Expected 1 bad row, found {failed_mask.sum()}"
retry_row = df[failed_mask].iloc[0]

subject_id = retry_row['subject_id']
label_int  = retry_row['label']
label_str  = retry_row['label_str']
exercise   = retry_row['exercise']
clip_name  = retry_row['clip_name']

print(f"Retrying: {clip_name}  ({label_str} / {exercise})")
print(f"  Correct clip_dir: {CORRECT_CLIP_DIR}")

result = process_one_clip(CORRECT_CLIP_DIR, subject_id, label_int, label_str, exercise, face_mesh)

if result is None:
    print("❌ Still failed — check that the folder has .jpg/.png images")
else:
    # Save per-clip files
    clip_features_dir = os.path.join(FEATURES_OUTPUT_DIR, f"{label_str}_{exercise}_{clip_name}")
    os.makedirs(clip_features_dir, exist_ok=True)

    result['xyz_df'].to_csv(os.path.join(clip_features_dir, f"{clip_name}_landmarks_xyz.csv"), index=False)
    result['distances_df'].to_csv(os.path.join(clip_features_dir, f"{clip_name}_distances.csv"), index=False)
    result['abs_diff_df'].to_csv(os.path.join(clip_features_dir, f"{clip_name}_abs_diff.csv"), index=False)
    result['abs_ratio_df'].to_csv(os.path.join(clip_features_dir, f"{clip_name}_abs_ratio.csv"), index=False)
    np.save(os.path.join(clip_features_dir, 'stream2_features.npy'), result['stream2_arr'])
    np.save(os.path.join(clip_features_dir, 'landmarks_478.npy'), result['landmarks_arr'])

    # Add to task_dfs so combined CSVs are up-to-date
    for key, df_data in [("distances", result['distances_df']),
                          ("abs_diff",  result['abs_diff_df']),
                          ("abs_ratio", result['abs_ratio_df'])]:
        df_copy = df_data.copy()
        df_copy['subject_id'] = subject_id
        df_copy['label'] = label_int
        df_copy['task'] = exercise
        task_dfs[key].setdefault(exercise, []).append(df_copy)

    combined_df = result['distances_df'].copy()
    for col in asym_diff_labels:
        if col in result['abs_diff_df'].columns:
            combined_df[col] = result['abs_diff_df'][col]
    for col in asym_ratio_labels:
        if col in result['abs_ratio_df'].columns:
            combined_df[col] = result['abs_ratio_df'][col]
    combined_df['subject_id'] = subject_id
    combined_df['label'] = label_int
    combined_df['task'] = exercise
    task_dfs['combined'].setdefault(exercise, []).append(combined_df)

    print(f"✅ Saved to: {clip_features_dir}")
    print(f"   Stream2 array shape: {result['stream2_arr'].shape}  (frames, 76)")
    print(f"\nRe-run cells 'save-taskwise' and 'save-overall' to update the combined CSVs.")

---
## 6. Save Task-Wise Combined CSVs

Output matches your existing format: `BIGSMILE_abs_diff_only.csv`, `BLOW_abs_diff_only.csv`, etc.

In [ ]:
# ── Save task-wise combined CSVs ──
task_wise_dir = os.path.join(OUTPUT_DIR, "TASK_WISE_COMBINED")
os.makedirs(task_wise_dir, exist_ok=True)

print("Saving task-wise combined CSVs...")

for file_type in ["distances", "abs_diff", "abs_ratio", "combined"]:
    for task_name, df_list in task_dfs[file_type].items():
        combined = pd.concat(df_list, ignore_index=True)
        save_path = os.path.join(task_wise_dir, f"{task_name}_{file_type}.csv")
        combined.to_csv(save_path, index=False)
        print(f"  ✅ {task_name}_{file_type}.csv  ({combined.shape[0]} rows, {combined.shape[1]} cols)")

print(f"\nAll task-wise files saved to: {task_wise_dir}")

Saving task-wise combined CSVs...
  ✅ BBP_distances.csv  (3917 rows, 42 cols)
  ✅ BIG_SMILE_distances.csv  (421 rows, 42 cols)
  ✅ BLOW_distances.csv  (1066 rows, 42 cols)
  ✅ BROW_distances.csv  (721 rows, 42 cols)
  ✅ KISS_distances.csv  (1600 rows, 42 cols)
  ✅ OPEN_distances.csv  (1692 rows, 42 cols)
  ✅ PA_distances.csv  (2215 rows, 42 cols)
  ✅ PATAKA_distances.csv  (1952 rows, 42 cols)
  ✅ SPREAD_distances.csv  (1881 rows, 42 cols)
  ✅ BBP_abs_diff.csv  (3917 rows, 24 cols)
  ✅ BIG_SMILE_abs_diff.csv  (421 rows, 24 cols)
  ✅ BLOW_abs_diff.csv  (1066 rows, 24 cols)
  ✅ BROW_abs_diff.csv  (721 rows, 24 cols)
  ✅ KISS_abs_diff.csv  (1600 rows, 24 cols)
  ✅ OPEN_abs_diff.csv  (1692 rows, 24 cols)
  ✅ PA_abs_diff.csv  (2215 rows, 24 cols)
  ✅ PATAKA_abs_diff.csv  (1952 rows, 24 cols)
  ✅ SPREAD_abs_diff.csv  (1881 rows, 24 cols)
  ✅ BBP_abs_ratio.csv  (3917 rows, 24 cols)
  ✅ BIG_SMILE_abs_ratio.csv  (421 rows, 24 cols)
  ✅ BLOW_abs_ratio.csv  (1066 rows, 24 cols)
  ✅ BROW_abs_ratio.

---
## 7. Save Overall Combined CSV (all exercises, all subjects)

In [ ]:
# ── Overall combined files ──
print("Saving overall combined CSVs...")

for file_type in ["distances", "abs_diff", "abs_ratio", "combined"]:
    all_dfs = []
    for task_name, df_list in task_dfs[file_type].items():
        all_dfs.extend(df_list)
    
    if all_dfs:
        overall = pd.concat(all_dfs, ignore_index=True)
        save_path = os.path.join(OUTPUT_DIR, f"OVERALL_{file_type}.csv")
        overall.to_csv(save_path, index=False)
        print(f"  ✅ OVERALL_{file_type}.csv  ({overall.shape[0]} rows, {overall.shape[1]} cols)")
        
        # Quick stats
        print(f"     HC frames: {(overall['label']==0).sum()}, Stroke frames: {(overall['label']==1).sum()}")
        print(f"     Tasks: {sorted(overall['task'].unique())}")
        print(f"     Subjects: {overall['subject_id'].nunique()}")

print(f"\nAll overall files saved to: {OUTPUT_DIR}")

Saving overall combined CSVs...
  ✅ OVERALL_distances.csv  (15465 rows, 42 cols)
     HC frames: 6728, Stroke frames: 8737
     Tasks: ['BBP', 'BIG_SMILE', 'BLOW', 'BROW', 'KISS', 'OPEN', 'PA', 'PATAKA', 'SPREAD']
     Subjects: 22
  ✅ OVERALL_abs_diff.csv  (15465 rows, 24 cols)
     HC frames: 6728, Stroke frames: 8737
     Tasks: ['BBP', 'BIG_SMILE', 'BLOW', 'BROW', 'KISS', 'OPEN', 'PA', 'PATAKA', 'SPREAD']
     Subjects: 22
  ✅ OVERALL_abs_ratio.csv  (15465 rows, 24 cols)
     HC frames: 6728, Stroke frames: 8737
     Tasks: ['BBP', 'BIG_SMILE', 'BLOW', 'BROW', 'KISS', 'OPEN', 'PA', 'PATAKA', 'SPREAD']
     Subjects: 22
  ✅ OVERALL_combined.csv  (15465 rows, 82 cols)
     HC frames: 6728, Stroke frames: 8737
     Tasks: ['BBP', 'BIG_SMILE', 'BLOW', 'BROW', 'KISS', 'OPEN', 'PA', 'PATAKA', 'SPREAD']
     Subjects: 22

All overall files saved to: C:\Users\Nicola Sambo\Desktop\Thesis\Data\TORONTO_DATA\Paper_04\FACE_PRECOCESSING\preprocessing_pipeline\RESULTS_FROM_PRE_5_ffps\fusion\STREA

---
## 8. Verify Everything

In [ ]:
# ── Verify all clips have their feature files ──
# Files are saved in FEATURES_OUTPUT_DIR/per_clip_features/{label_str}_{exercise}_{clip_name}/
# NOT in the original clip_dir — that's the source images folder.

print("Verification:")

missing_s2 = []
missing_lm = []
shapes = []

for _, row in df.iterrows():
    clip_name  = row['clip_name']
    label_str  = row['label_str']
    exercise   = row['exercise']
    clip_feat_dir = os.path.join(FEATURES_OUTPUT_DIR, "per_clip_features",
                                 f"{label_str}_{exercise}_{clip_name}")
    s2_path = os.path.join(clip_feat_dir, 'stream2_features.npy')
    lm_path = os.path.join(clip_feat_dir, 'landmarks_478.npy')

    if not os.path.exists(s2_path):
        missing_s2.append(f"{label_str}_{exercise}_{clip_name}")
    else:
        arr = np.load(s2_path)
        shapes.append(arr.shape)

    if not os.path.exists(lm_path):
        missing_lm.append(f"{label_str}_{exercise}_{clip_name}")

print(f"  stream2_features.npy : {len(df) - len(missing_s2)}/{len(df)} clips ✅")
print(f"  landmarks_478.npy    : {len(df) - len(missing_lm)}/{len(df)} clips ✅")

if shapes:
    print(f"\n  Shape check (first 5):")
    for s in shapes[:5]:
        status = '✅' if s[1] == 76 else '❌ (expected 76 features)'
        print(f"    {s}  {status}")

if missing_s2:
    print(f"\n  ⚠️ Missing stream2_features ({len(missing_s2)} clips):")
    for m in missing_s2[:5]:
        print(f"    {m}")

print(f"\n{'='*60}")
print(f"  FEATURE OUTPUT LOCATION")
print(f"{'='*60}")
print(f"  {FEATURES_OUTPUT_DIR}/per_clip_features/")
print(f"    {{label_str}}_{{exercise}}_{{clip_name}}/")
print(f"      stream2_features.npy   ← (N_frames, 76)")
print(f"      landmarks_478.npy      ← (N_frames, 478, 3)")
print(f"      *_landmarks_xyz.csv")
print(f"      *_distances.csv")
print(f"      *_abs_diff.csv")
print(f"      *_abs_ratio.csv")